# **Expected Gradients: the baseline as a distribution**

Practice for the module [«Attribution from axioms: IG, DeepLIFT, LRP»](https://ai-interpretability.school).

The lesson on choosing a baseline reached an uncomfortable conclusion: **the baseline is the
question your explanation answers**, and any single point has a blind spot. A black baseline
loses everything dark, a white one everything light.

The lesson recommends Expected Gradients, where the baseline is a distribution rather than
a point. Here we check that by hand:

- we see the blind spot of the black baseline **as a number**, not as a claim;
- we implement Expected Gradients in ten lines and watch the spot disappear;
- we check completeness for both — the sum of attributions against the difference of predictions.

In [ ]:
import torch
import numpy as np
import requests
import matplotlib.pyplot as plt
from io import BytesIO
from PIL import Image
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights

torch.manual_seed(0);

In [ ]:
model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model.eval();

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def load(name):
    return Image.open(BytesIO(requests.get('https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/' + name).content)).convert('RGB')

image = load('hog.jpg')
x = transform(image).unsqueeze(0)

categories = [s.strip() for s in requests.get('https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/imagenet_classes.txt').text.splitlines()]
with torch.no_grad():
    target = model(x).argmax().item()
print('Class:', target, categories[target])

## 1. Integrated Gradients with a single baseline

The formula from the lesson, approximated by a Riemann sum:

$$\text{IG}_i(x) = (x_i - x'_i)\times\frac1m\sum_{k=1}^{m}
\frac{\partial F\big(x' + \tfrac{k}{m}(x-x')\big)}{\partial x_i}$$

The factor $(x_i - x'_i)$ stands in front of the integral — and it is what creates the blind
spot: wherever the input coincides with the baseline the attribution vanishes, however large
the gradient.

In [ ]:
def grads_at(model, points, target):
    """Gradients of the class logit at every point of the path."""
    points = points.clone().requires_grad_(True)
    logits = model(points)[:, target].sum()
    g, = torch.autograd.grad(logits, points)
    return g

def integrated_gradients(model, x, baseline, target, m=32):
    alphas = torch.linspace(1 / m, 1.0, m).view(-1, 1, 1, 1)
    path = baseline + alphas * (x - baseline)
    g = grads_at(model, path, target).mean(0, keepdim=True)
    return (x - baseline) * g

# Careful: `torch.zeros_like(x)` is NOT a black image. The input is already normalized,
# and zeros in normalized coordinates correspond to the mean ImageNet brightness, i.e. grey.
# A black baseline comes from running a black image through the same transform.
black = transform(Image.new('RGB', (224, 224), (0, 0, 0))).unsqueeze(0)
grey = torch.zeros_like(x)

ig_black = integrated_gradients(model, x, black, target)
ig_grey = integrated_gradients(model, x, grey, target)
print('black baseline in normalized coordinates:', [round(v, 2) for v in black[0, :, 0, 0].tolist()])
print('while zeros correspond to the brightness:', [round(v, 2) for v in (grey[0, :, 0, 0] * 0 + 0.449).tolist()])

## 2. The blind spot, as a number

Take a mask of dark pixels (in the original scale, before normalization) and see what share of
the whole «mass» of the attribution falls on them. If the method looks at dark regions honestly,
the share should be comparable to their area.

In [ ]:
raw = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])(image)
dark = (raw.mean(0) < 0.25)          # mask of the dark pixels
print(f'dark pixels take up {dark.float().mean():.1%} of the area')

def dark_share(attr):
    a = attr.abs().sum(1)[0]         # channels collapsed
    return (a[dark].sum() / a.sum()).item()

print(f'IG, black baseline: the dark areas get {dark_share(ig_black):.1%} of the attribution')
print(f'IG, grey  baseline: the dark areas get {dark_share(ig_grey):.1%} of the attribution')

**Task 1.** Build IG with a **white** baseline and compute the same share. Where has the blind
spot moved?

In [ ]:
# Your code here

## 3. Expected Gradients

The baseline becomes a distribution: at every step we take a random object from the dataset and
a random point on the path towards it.

$$\text{EG}_i(x) = \mathbb{E}_{x'\sim D,\ \alpha\sim U(0,1)}
\Big[(x_i - x'_i)\cdot\frac{\partial F\big(x' + \alpha(x - x')\big)}{\partial x_i}\Big]$$

Note that this is exactly the IG expression, only the expectation is taken over the baseline as
well. One sample per object instead of $m$ integration steps.

In [ ]:
# A «dataset» of baselines: other course images. In your own task use the training set.
pool = torch.cat([transform(load(n)).unsqueeze(0)
                  for n in ('cat.jpg', 'cat_and_dog.jpg', 'pig.png')])
print('objects in the baseline pool:', pool.shape[0])

def expected_gradients(model, x, pool, target, n=64):
    idx = torch.randint(0, pool.shape[0], (n,))
    baselines = pool[idx]
    alphas = torch.rand(n, 1, 1, 1)
    points = baselines + alphas * (x - baselines)
    g = grads_at(model, points, target)
    return ((x - baselines) * g).mean(0, keepdim=True)

eg = expected_gradients(model, x, pool, target)
print(f'Expected Gradients: the dark areas get {dark_share(eg):.1%} of the attribution')

**Task 2.** Compare three numbers: the area share of the dark pixels, the attribution share for
IG with a black baseline, and the same for Expected Gradients. Which of the two explanations
loses the dark regions?

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(raw.permute(1, 2, 0)); ax[0].set_title('original')
for a, (m, t) in zip(ax[1:], [(ig_black, 'IG, black baseline'), (eg, 'Expected Gradients')]):
    a.imshow(m.abs().sum(1)[0].detach().numpy(), cmap='hot')
    a.set_title(t)
for a in ax:
    a.axis('off')
plt.tight_layout()
plt.show()

## 4. Completeness: the sum of attributions against the difference of predictions

The completeness axiom promises $\sum_i \text{IG}_i(x) = F(x) - F(x')$. Checking it is one line,
and this is exactly how a too coarse approximation of the integral is caught (in Captum this
residual is called the convergence delta).

In [ ]:
with torch.no_grad():
    fx = model(x)[0, target].item()
    f_black = model(black)[0, target].item()
    f_pool = model(pool)[:, target].mean().item()

print(f'{"method":28s} {"sum of attributions":>20s} {"F(x) − F(x′)":>14s} {"residual":>10s}')
for name, attr, base in (('IG, black baseline', ig_black, f_black),
                         ('Expected Gradients', eg, f_pool)):
    s = attr.sum().item()
    print(f'{name:28s} {s:20.3f} {fx - base:14.3f} {abs(s - (fx - base)):10.3f}')

**Task 3.** Raise the number of IG steps from 32 to 128 and the number of EG samples from 64
to 256. How does the residual change for each? For which one does it fall faster?

**Task 4.** Coinciding with the baseline zeroes the attribution — check it directly: zero out
a 60×60 square in the image and compute the sum of absolute attributions inside it for IG with
a black baseline. The expected answer is exactly zero.

In [ ]:
# Your code here

## What to take away

- **The blind spot is measurable.** Not «a black baseline loses the dark» but a concrete share of
  the attribution that takes three lines to compute and can go into a report.
- **Expected Gradients is the same IG**, with the expectation taken over the baseline as well.
  Almost the same amount of code, and no blind spot of a single point.
- **Always check completeness.** A noticeable residual means the number of steps is too small and
  the integral has not converged — not that the method is bad.
- **State the baseline next to the picture.** A map without one is about as informative as
  a confidence interval without a confidence level.